# Step 1: Wyscout ↔ Transfermarkt join, transfer funnel, baselines

What this notebook does, in order:
1. Builds 2017/18 player stats from Wyscout (minutes from line-ups, per-90 event counts).
2. Links Wyscout players to Transfermarkt players (date of birth + fuzzy name, then a stricter fallback).
3. Turns each linked player's next club-to-club move into one example with label
   **y = minutes played for the new club / (90 × that club's matches) in the 365 days after the transfer**.
4. Prints the **funnel** (how many transfers survive each filter): this is your data section.
5. Runs the constant-mean and linear-regression baselines on a fixed held-out split.


In [21]:
import os, re, json, glob, gc
import numpy as np
import pandas as pd
from unidecode import unidecode
from rapidfuzz import fuzz
try:
    from IPython.display import display
except ImportError:
    display = print

from pathlib import Path
# Walk up from the working directory to the repo root (the folder containing data/raw),
# so this works whether Jupyter starts in the repo root, notebooks/, or any other subfolder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No data/raw folder found in {Path.cwd()} or its parents")
WY_DIR = str(ROOT / "data/raw/wyscout")
TM_DIR = str(ROOT / "data/raw/transfermarkt")
OUT = ROOT / "data/processed"
print("Wyscout dir:", WY_DIR)
print("Transfermarkt dir:", TM_DIR)

LEAGUES = ["England", "France", "Germany", "Italy", "Spain"]  # 2017/18 club seasons (World Cup / Euro skipped)

# ---- dataset definition (defaults; the sensitivity cell varies these) ----
WINDOW_START   = "2018-05-15"  # after the Wyscout season ends, so features are strictly pre-transfer
WINDOW_END     = "2019-02-15"  # summer 2018 + winter 2019 transfer windows
LABEL_DAYS     = 365
MIN_DEST_GAMES = 10            # destination club needs >=10 Transfermarkt-covered games in the label window
MIN_WY_MINUTES = 450           # >=5 full matches of Wyscout events, so per-90 stats aren't noise
SEED = 42
pd.set_option("display.width", 160)

Wyscout dir: /home/slage/Coding/Deep Learning/Project/Deep-Learning-Scouting/data/raw/wyscout
Transfermarkt dir: /home/slage/Coding/Deep Learning/Project/Deep-Learning-Scouting/data/raw/transfermarkt


In [22]:
def read_tm(name):
    for ext in (".csv", ".csv.gz"):
        hits = glob.glob(os.path.join(TM_DIR, "**", name + ext), recursive=True)
        if hits:
            return pd.read_csv(hits[0], low_memory=False)
    raise FileNotFoundError(f"{name}.csv(.gz) not found under {TM_DIR}")

def find_wy(fname):
    hits = glob.glob(os.path.join(WY_DIR, "**", fname), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{fname} not found under {WY_DIR}")
    return hits[0]

def load_json(fname):
    with open(find_wy(fname), encoding="utf-8") as fh:
        return json.load(fh)

def fix_text(s):
    """Some Wyscout names arrive with literal '\\u00e9' escapes; decode them."""
    if not isinstance(s, str):
        return ""
    if "\\u" in s:
        try:
            s = s.encode("latin-1", "backslashreplace").decode("unicode_escape")
        except Exception:
            pass
    return s

def norm_name(s):
    s = unidecode(fix_text(s)).lower()
    s = re.sub(r"[^a-z ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

## 1. Wyscout: players, minutes from line-ups, per-90 stats

In [23]:
wy_players = pd.DataFrame(load_json("players.json"))
for c in ["firstName", "middleName", "lastName", "shortName"]:
    wy_players[c] = wy_players[c].map(fix_text) if c in wy_players else ""
wy_players["dob"] = pd.to_datetime(wy_players["birthDate"], errors="coerce").dt.normalize()
wy_players["wy_role"] = wy_players["role"].map(lambda r: r.get("code2") if isinstance(r, dict) else None)
wy_players = wy_players.rename(columns={"wyId": "wy_id"})[
    ["wy_id", "firstName", "middleName", "lastName", "shortName", "dob", "wy_role"]]

wy_teams = pd.DataFrame(load_json("teams.json"))
WY_TEAM_NAME = dict(zip(wy_teams["wyId"], wy_teams["name"].map(fix_text)))

def match_minutes(m):
    """Minutes per player in one match from the line-up and substitutions (red cards ignored)."""
    rows = []
    for tid, td in m["teamsData"].items():
        form = td.get("formation") or {}
        start = {p["playerId"]: 0 for p in (form.get("lineup") or [])}
        end = {p: 90 for p in start}
        subs = form.get("substitutions")
        if isinstance(subs, list):  # Wyscout stores the string "null" when there were none
            for s in subs:
                minute = min(int(s["minute"]), 90)
                if s["playerOut"] in end:
                    end[s["playerOut"]] = minute
                start[s["playerIn"]] = minute
                end.setdefault(s["playerIn"], 90)
        for p in start:
            rows.append((p, int(tid), m["wyId"], max(end[p] - start[p], 0)))
    return rows

rows = []
for lg in LEAGUES:
    for m in load_json(f"matches_{lg}.json"):
        rows += [r + (lg,) for r in match_minutes(m)]
wy_min = pd.DataFrame(rows, columns=["wy_id", "team_id", "match_id", "minutes", "league"])
wy_min = wy_min[wy_min.wy_id != 0]  # playerId 0 = unknown player in some substitution records, so ignore it
team_matches = wy_min.groupby("team_id")["match_id"].nunique()

main_team = (wy_min.groupby(["wy_id", "team_id", "league"])["minutes"].sum().reset_index()
             .sort_values("minutes", ascending=False).drop_duplicates("wy_id").set_index("wy_id"))
wy = wy_min.groupby("wy_id")["minutes"].sum().rename("wy_minutes").to_frame()
wy = wy.join(main_team[["team_id", "league"]])
wy = wy[wy.wy_minutes > 0].reset_index()
wy["wy_share"] = (wy.wy_minutes / (90 * wy.team_id.map(team_matches))).clip(upper=1)
wy["wy_team"] = wy.team_id.map(WY_TEAM_NAME)
wy = wy.merge(wy_players, on="wy_id", how="left")
print(f"Wyscout: {len(wy_players):,} players in players.json, "
      f"{len(wy):,} played league minutes in 2017/18")
print(wy.league.value_counts().to_string())

Wyscout: 3,603 players in players.json, 2,561 played league minutes in 2017/18
league
Spain      552
France     525
Italy      523
England    497
Germany    464


In [24]:
def player_event_counts(path):
    with open(path, encoding="utf-8") as fh:
        ev = pd.DataFrame(json.load(fh))[["playerId", "eventName", "subEventName", "tags"]]
    ev = ev[ev.playerId != 0]
    tags = ev["tags"].map(lambda t: {d["id"] for d in t})
    has = lambda tag: tags.map(lambda s: tag in s)
    is_pass = ev.eventName.eq("Pass")
    is_shot = ev.eventName.eq("Shot") | ev.subEventName.isin(["Penalty", "Free kick shot"])
    is_duel = ev.eventName.eq("Duel")
    out = pd.DataFrame({
        "wy_id": ev.playerId.values,
        "passes": is_pass.values,
        "passes_acc": (is_pass & has(1801)).values,      # 1801 = accurate
        "key_passes": has(302).values,
        "assists": has(301).values,
        "shots": is_shot.values,
        "goals": (is_shot & has(101)).values,            # 101 = goal
        "duels": is_duel.values,
        "duels_won": (is_duel & has(703)).values,        # 703 = won
        "interceptions": has(1401).values,
        "fouls": ev.eventName.eq("Foul").values,
    }).groupby("wy_id").sum()
    del ev, tags
    gc.collect()
    return out

counts = []
for lg in LEAGUES:  # one file at a time to keep Colab RAM under control
    counts.append(player_event_counts(find_wy(f"events_{lg}.json")))
    print("events loaded:", lg)
counts = pd.concat(counts).groupby(level=0).sum()
STATS = list(counts.columns)

wy = wy.merge(counts, left_on="wy_id", right_index=True, how="left")
wy[STATS] = wy[STATS].fillna(0)
P90 = []
for c in STATS:
    if c != "passes_acc":
        wy[c + "_p90"] = wy[c] / wy.wy_minutes * 90
        P90.append(c + "_p90")
wy["pass_acc"] = wy.passes_acc / wy.passes.replace(0, np.nan)
wy[["wy_minutes", "wy_share"] + P90].describe().T.round(2)

events loaded: England
events loaded: France
events loaded: Germany
events loaded: Italy
events loaded: Spain


,count,mean,std,min,25%,50%,75%,max
wy_minutes,2561.0,1411.71,989.21,1.00,478.00,1362.00,2238.00,3420.00
wy_share,2561.0,0.42,0.29,0.00,0.14,0.41,0.67,1.00
passes_p90,2561.0,38.31,17.16,3.21,26.38,37.01,46.69,225.00
key_passes_p90,2561.0,0.28,0.47,0.00,0.00,0.15,0.39,12.86
assists_p90,2561.0,0.07,0.13,0.00,0.00,0.00,0.10,2.90
shots_p90,2561.0,1.13,1.65,0.00,0.24,0.75,1.69,45.00
goals_p90,2561.0,0.12,0.63,0.00,0.00,0.00,0.14,30.00
duels_p90,2561.0,20.37,11.71,0.00,15.47,19.87,24.72,270.00
duels_won_p90,2561.0,7.61,4.37,0.00,5.70,7.40,9.22,90.00
interceptions_p90,2561.0,3.98,2.79,0.00,2.16,3.89,5.50,90.00


## 2. Transfermarkt tables

In [25]:
tm_players = read_tm("players")
transfers  = read_tm("transfers")
apps       = read_tm("appearances")
games      = read_tm("games")
clubs      = read_tm("clubs")
vals       = read_tm("player_valuations")
for n, d in [("players", tm_players), ("transfers", transfers), ("appearances", apps),
             ("games", games), ("clubs", clubs), ("player_valuations", vals)]:
    print(f"{n:18s} {str(d.shape):16s} {list(d.columns)}")

tm_players["dob"] = pd.to_datetime(tm_players["date_of_birth"], errors="coerce").dt.normalize()
tm_players["nm"] = tm_players["name"].map(norm_name)
apps["date"] = pd.to_datetime(apps["date"], errors="coerce")
games["date"] = pd.to_datetime(games["date"], errors="coerce")
vals["date"] = pd.to_datetime(vals["date"], errors="coerce")
transfers["transfer_date"] = pd.to_datetime(transfers["transfer_date"], errors="coerce")
CLUB_NAME = dict(zip(clubs["club_id"], clubs["name"]))
CLUB_COMP = dict(zip(clubs["club_id"], clubs.get("domestic_competition_id", pd.Series(dtype=str))))
TM_DOB = tm_players.set_index("player_id")["dob"]
TM_POS = tm_players.set_index("player_id")["position"]

players            (50149, 26)      ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']
transfers          (175165, 10)     ['player_id', 'transfer_date', 'transfer_season', 'from_club_id', 'to_club_id', 'from_club_name', 'to_club_name', 'transfer_fee', 'market_value_in_eur', 'player_name']
appearances        (1894350, 13)    ['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']
games              (88958, 23)      ['game_id', 'c

## 3. Player join
Pass 1: same date of birth and a name similarity ≥ 80 (checked against several Wyscout name variants,
so "Neymar da Silva Santos Júnior" still finds "Neymar").
Pass 2, for Wyscout players left over: same birth year, a stricter full-name match ≥ 90, and the
Transfermarkt player must have played in 2017/18. Both passes need a clear winner (5-point gap to the
runner-up) and every Transfermarkt ID is used once.

In [26]:
def name_variants(r):
    v = {norm_name(f"{r.firstName} {r.lastName}"),
         norm_name(f"{r.firstName} {r.middleName} {r.lastName}"),
         norm_name(re.sub(r"^\w\.\s*", "", r.shortName or "")),  # "L. Messi" -> "messi"
         norm_name(r.lastName)}
    return [x for x in v if x]

wy["variants"] = [name_variants(r) for r in wy.itertuples()]

def best_unique(c, margin=5):
    c = c.sort_values(["wy_id", "score"], ascending=[True, False]).copy()
    c["rk"] = c.groupby("wy_id").cumcount()
    best = c[c.rk == 0].set_index("wy_id")
    second = c[c.rk == 1].set_index("wy_id")["score"].reindex(best.index).fillna(0)
    best = best[(best.score - second) >= margin].reset_index()
    return best.sort_values("score", ascending=False).drop_duplicates("player_id")[["wy_id", "player_id", "score"]]

# pass 1: date of birth + name
c1 = (wy[["wy_id", "dob", "variants"]].dropna(subset=["dob"])
      .merge(tm_players[["player_id", "dob", "nm"]].dropna(subset=["dob"]), on="dob"))
c1["score"] = [max((fuzz.token_set_ratio(v, n) for v in vs), default=0) for vs, n in zip(c1.variants, c1.nm)]
m1 = best_unique(c1[c1.score >= 80]).assign(match_pass="dob+name")

# pass 2: birth year + strict full name + active in 2017/18
season_1718 = apps[(apps.date >= "2017-07-01") & (apps.date < "2018-07-01")]
active = tm_players[tm_players.player_id.isin(season_1718.player_id) & ~tm_players.player_id.isin(m1.player_id)]
active = active.assign(yr=active.dob.dt.year)[["player_id", "yr", "nm"]].dropna(subset=["yr"])
left = wy[~wy.wy_id.isin(m1.wy_id)][["wy_id", "dob", "variants"]]
left = left.assign(yr=left.dob.dt.year).drop(columns="dob").dropna(subset=["yr"])
c2 = left.merge(active, on="yr")
c2["score"] = [max((fuzz.token_sort_ratio(v, n) for v in vs if " " in v), default=0)
               for vs, n in zip(c2.variants, c2.nm)]
m2 = best_unique(c2[c2.score >= 90]).assign(match_pass="year+name+active")

link = pd.concat([m1, m2], ignore_index=True)
link["player_id"] = link.player_id.astype(int)
wy = wy.drop(columns=[c for c in ["player_id", "score", "match_pass"] if c in wy]).merge(link, on="wy_id", how="left")

n_link = wy.player_id.notna().sum()
print(f"Matched {n_link:,} / {len(wy):,} Wyscout league players ({n_link / len(wy):.1%})")
print(wy.match_pass.value_counts().to_string())
print("\nMatch rate by league:")
print(wy.groupby("league").player_id.apply(lambda s: s.notna().mean()).round(3).to_string())

Matched 2,523 / 2,561 Wyscout league players (98.5%)
match_pass
dob+name            2473
year+name+active      50

Match rate by league:
league
England    0.986
France     0.989
Germany    0.985
Italy      0.996
Spain      0.971


In [27]:
# Precision check: does the matched player's main 2017/18 club agree across the two sources?
# Disagreements are either January movers (fine) or wrong matches (fix or raise thresholds).
tm_club_1718 = (season_1718.groupby(["player_id", "player_club_id"])["minutes_played"].sum().reset_index()
                .sort_values("minutes_played", ascending=False).drop_duplicates("player_id")
                .set_index("player_id")["player_club_id"].map(CLUB_NAME))
mt = wy[wy.player_id.notna()].copy()
mt["tm_name"] = mt.player_id.map(tm_players.set_index("player_id")["name"])
mt["tm_club"] = mt.player_id.map(tm_club_1718)
mt["club_agree"] = [float(fuzz.token_set_ratio(norm_name(a), norm_name(b)) >= 70) if isinstance(b, str) else np.nan
                    for a, b in zip(mt.wy_team, mt.tm_club)]
print(f"Club agreement among matched players with a 2017/18 TM club: {mt.club_agree.mean():.1%}")
cols = ["firstName", "lastName", "wy_team", "tm_name", "tm_club", "score", "match_pass"]
print("\nRandom sample of matches (eyeball these):")
display(mt.sample(min(15, len(mt)), random_state=SEED)[cols])
print("\nClub disagreements:")
display(mt[mt.club_agree == 0][cols].head(20))

Club agreement among matched players with a 2017/18 TM club: 95.2%

Random sample of matches (eyeball these):


,firstName,lastName,wy_team,tm_name,tm_club,score,match_pass
1954,Takuma,Asano,Stuttgart,Takuma Asano,VfB Stuttgart,100.0,dob+name
300,Carles,Planas Antolínez,Girona,Carles Planas,Girona FC,100.0,dob+name
494,James,McArthur,Crystal Palace,James McArthur,Crystal Palace,100.0,dob+name
213,Mikel,Balenziaga Oruesagasti,Athletic Club,Mikel Balenziaga,Athletic Bilbao,100.0,dob+name
488,Mark,Noble,West Ham United,Mark Noble,West Ham United,100.0,dob+name
2362,Riccardo,Orsolini,Bologna,Riccardo Orsolini,Bologna Football Club 1909,100.0,dob+name
1509,Simeon,Tochukwu Nwankwo,Crotone,Simy,FC Crotone,100.0,dob+name
947,Riccardo,Meggiorini,Chievo,Riccardo Meggiorini,Chievo Verona,100.0,dob+name
1738,Héctor,Bellerín Moruno,Arsenal,Héctor Bellerín,Arsenal FC,100.0,dob+name
1850,Adam,Marušić,Lazio,Adam Marusic,Società Sportiva Lazio S.p.A.,100.0,dob+name



Club disagreements:


,firstName,lastName,wy_team,tm_name,tm_club,score,match_pass
19,Jürgen,Locadia,Brighton & Hove Albion,Jürgen Locadia,PSV Eindhoven,100.0,dob+name
49,Viktor,Fischer,Mainz 05,Viktor Fischer,FC Copenhagen,100.0,dob+name
75,Ángel Fabián,di María Hernández,PSG,Ángel Di María,Paris Saint-Germain,100.0,dob+name
76,Lassana,Diarra,PSG,Lassana Diarra,Paris Saint-Germain,100.0,dob+name
89,Daniel,Alves da Silva,PSG,Dani Alves,Paris Saint-Germain,100.0,dob+name
90,Marc,Bartra Aregall,Real Betis,Marc Bartra,Borussia Dortmund,100.0,dob+name
97,Rafael,Alcântara do Nascimento,Internazionale,Rafinha,Inter Milan,100.0,dob+name
129,João,Miranda de Souza Filho,Internazionale,Miranda,Inter Milan,100.0,dob+name
165,Borja,Valero Iglesias,Internazionale,Borja Valero,Inter Milan,100.0,dob+name
284,Víctor,Machín Pérez,Las Palmas,Vitolo,Atlético de Madrid,100.0,dob+name


## 4. Label: post-transfer playing-time share
`time_share(player, club, start, end)` = minutes for that club in [start, end) / (90 × club games in
[start, end)). The denominator counts every game Transfermarkt has for the club (league, cups, Europe),
capped at 1. Clubs in leagues Transfermarkt doesn't cover have no games → no label → dropped, which is
honest rather than a fake 0.

In [28]:
cg = pd.concat([games[["home_club_id", "date"]].set_axis(["club_id", "date"], axis=1),
                games[["away_club_id", "date"]].set_axis(["club_id", "date"], axis=1)]).dropna()
CLUB_DATES = {int(c): np.sort(g["date"].values) for c, g in cg.groupby("club_id")}
apps_l = apps[apps.player_id.isin(set(link.player_id))].dropna(subset=["date"])
APPS = {(int(p), int(c)): (g["date"].values, g["minutes_played"].fillna(0).values)
        for (p, c), g in apps_l.groupby(["player_id", "player_club_id"])}
EMPTY = (np.array([], dtype="datetime64[ns]"), np.array([]))

def time_share(pid, club, start, end):
    if pd.isna(club):
        return np.nan, 0
    d = CLUB_DATES.get(int(club))
    if d is None:
        return np.nan, 0
    s, e = start.to_datetime64(), end.to_datetime64()
    n = int(np.searchsorted(d, e) - np.searchsorted(d, s))
    if n == 0:
        return np.nan, 0
    dates, mins = APPS.get((int(pid), int(club)), EMPTY)
    return min(mins[(dates >= s) & (dates < e)].sum() / (90 * n), 1.0), n

# Flag likely loan returns: player goes back to the club he left in his previous move, within ~13 months
tr = transfers.dropna(subset=["transfer_date"]).sort_values(["player_id", "transfer_date"]).copy()
g = tr.groupby("player_id")
tr["loan_return"] = ((tr.to_club_id == g.from_club_id.shift())
                     & ((tr.transfer_date - g.transfer_date.shift()).dt.days < 400))
vals_s = vals.dropna(subset=["date", "market_value_in_eur"]).sort_values("date")
WY_LINK = link[["wy_id", "player_id"]]
WY_FEAT = wy.drop(columns=["player_id", "variants", "score", "match_pass"])

## 5. Build the dataset and print the funnel (data section)

In [29]:
BAD_DEST = r"retired|without club|unknown|career break"

def build_dataset(window_start=WINDOW_START, window_end=WINDOW_END, min_dest_games=MIN_DEST_GAMES,
                  min_wy_minutes=MIN_WY_MINUTES, drop_loan_returns=True):
    f = {}
    f["Wyscout players with 2017/18 league minutes"] = len(wy)
    f["  linked to a Transfermarkt player"] = len(WY_LINK)
    t = tr.merge(WY_LINK, on="player_id")
    f["transfers of linked players (all years)"] = len(t)
    t = t[(t.transfer_date >= window_start) & (t.transfer_date < window_end)]
    f[f"  dated {window_start} to {window_end}"] = len(t)
    t = t[t.to_club_id.notna() & (t.to_club_id != t.from_club_id)
          & ~t.to_club_name.fillna("").str.contains(BAD_DEST, case=False)]
    f["  club-to-club moves"] = len(t)
    if drop_loan_returns:
        t = t[~t.loan_return]
        f["  excluding loan returns"] = len(t)
    t = t.sort_values("transfer_date").drop_duplicates("player_id").reset_index(drop=True)
    f["  first move per player"] = len(t)
    lab = [time_share(r.player_id, r.to_club_id, r.transfer_date, r.transfer_date + pd.Timedelta(days=LABEL_DAYS))
           for r in t.itertuples()]
    t["y"] = [x[0] for x in lab]
    t["dest_games"] = [x[1] for x in lab]
    t = t[t.dest_games >= min_dest_games].reset_index(drop=True)
    f[f"  destination club covered (>= {min_dest_games} games)"] = len(t)
    t["prev_share_tm"] = [time_share(r.player_id, r.from_club_id, r.transfer_date - pd.Timedelta(days=365),
                                     r.transfer_date)[0] for r in t.itertuples()]
    t = t.merge(WY_FEAT, on="wy_id")
    t = t[t.wy_minutes >= min_wy_minutes].reset_index(drop=True)
    f[f"  >= {min_wy_minutes} Wyscout minutes  ->  FINAL"] = len(t)

    # baseline features known at transfer time
    t["age"] = (t.transfer_date - t.player_id.map(TM_DOB)).dt.days / 365.25
    t["position"] = t.player_id.map(TM_POS).fillna("Unknown")
    mv = pd.merge_asof(t[["player_id", "transfer_date"]].reset_index().sort_values("transfer_date"),
                       vals_s[["player_id", "date", "market_value_in_eur"]].rename(columns={"date": "transfer_date"}),
                       on="transfer_date", by="player_id", direction="backward").set_index("index")
    t["log_mv"] = np.log1p(mv["market_value_in_eur"].reindex(t.index))
    t["dest_comp"] = t.to_club_id.map(CLUB_COMP)
    return t, f

def show_funnel(f):
    w = max(len(k) for k in f)
    for k, v in f.items():
        print(f"{k:<{w}}  {v:>7,}")

data, funnel = build_dataset()
show_funnel(funnel)
N = len(data)
print(f"\n>>> FINAL N = {N}: " + ("above 300, proceed." if N >= 300 else
      "BELOW 300. Check the sensitivity table below, and plan a smaller model."))
print("\nLabel y (post-transfer playing-time share):")
print(data.y.describe().round(3).to_string())
print(f"share of y == 0: {(data.y == 0).mean():.1%}")
print("\nDestination competitions:")
print(data.dest_comp.value_counts().head(12).to_string())

Wyscout players with 2017/18 league minutes    2,561
  linked to a Transfermarkt player             2,523
transfers of linked players (all years)       14,822
  dated 2018-05-15 to 2019-02-15                 795
  club-to-club moves                             791
  excluding loan returns                         522
  first move per player                          464
  destination club covered (>= 10 games)         361
  >= 450 Wyscout minutes  ->  FINAL              278

>>> FINAL N = 278: BELOW 300. Check the sensitivity table below, and plan a smaller model.

Label y (post-transfer playing-time share):
count    278.000
mean       0.483
std        0.279
min        0.000
25%        0.238
50%        0.530
75%        0.709
max        1.000
share of y == 0: 5.0%

Destination competitions:
dest_comp
ES1    63
IT1    55
GB1    54
FR1    40
L1     31
PO1     8
TR1     8
BE1     6
SC1     5
GR1     3
NL1     2
KR1     2


## 6. Sensitivity: how N changes with each choice
Note: extending to summer 2019 means the label window runs into the 2020 COVID break, and features are
two seasons old for those players. January 2018 moves are deliberately excluded because Wyscout events
after the move would leak into the features.

In [30]:
configs = [("default", {}),
           ("+ summer 2019 window", dict(window_end="2019-09-15")),
           ("min Wyscout minutes 270", dict(min_wy_minutes=270)),
           ("keep loan returns", dict(drop_loan_returns=False)),
           ("min dest games 5", dict(min_dest_games=5)),
           ("all relaxations", dict(window_end="2019-09-15", min_wy_minutes=270,
                                    drop_loan_returns=False, min_dest_games=5))]
display(pd.DataFrame([(name, len(build_dataset(**kw)[0])) for name, kw in configs], columns=["config", "N"]))

,config,N
0,default,278
1,+ summer 2019 window,450
2,min Wyscout minutes 270,302
3,keep loan returns,330
4,min dest games 5,283
5,all relaxations,522


## 7. Baselines: constant mean and linear regression
20% of transfers are held out once and the split is saved to `split.csv`. The neural model later must
use the same split. CV numbers come from the training 80% only.

In [31]:
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

NUM = ["age", "log_mv", "prev_share_tm", "wy_share", "pass_acc"] + P90
CAT = ["position"]
train_df, test_df = train_test_split(data, test_size=0.2, random_state=SEED)
data.assign(split=np.where(data.index.isin(test_df.index), "test", "train"))[
    ["player_id", "wy_id", "transfer_date", "to_club_id", "split"]].to_csv("split.csv", index=False)

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
models = {"constant mean (sanity)": DummyRegressor(strategy="mean"),
          "linear regression": make_pipeline(pre, LinearRegression())}
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)

rows = []
for name, mdl in models.items():
    cv_mse = -cross_val_score(mdl, train_df[NUM + CAT], train_df.y, cv=cv, scoring="neg_mean_squared_error")
    mdl.fit(train_df[NUM + CAT], train_df.y)
    test_mse = mean_squared_error(test_df.y, mdl.predict(test_df[NUM + CAT]))
    rows.append((name, cv_mse.mean(), cv_mse.std(), test_mse, np.sqrt(test_mse)))
results = pd.DataFrame(rows, columns=["model", "cv_mse_mean", "cv_mse_std", "test_mse", "test_rmse"])
print(f"train n = {len(train_df)}, test n = {len(test_df)}")
display(results.round(4))

lr = models["linear regression"]
coef = pd.Series(lr[-1].coef_, index=lr[0].get_feature_names_out()).sort_values(key=abs, ascending=False)
print("\nLinear regression coefficients (standardised features):")
print(coef.round(4).head(15).to_string())

train n = 222, test n = 56


,model,cv_mse_mean,cv_mse_std,test_mse,test_rmse
0,constant mean (sanity),0.0770,0.0109,0.0833,0.2887
1,linear regression,0.0829,0.0124,0.0843,0.2903



Linear regression coefficients (standardised features):
cat__position_Goalkeeper               0.1091
num__duels_p90                         0.1049
cat__position_Defender                -0.0636
num__log_mv                            0.0567
num__interceptions_p90                 0.0540
num__duels_won_p90                    -0.0424
cat__position_Midfield                -0.0344
num__pass_acc                          0.0277
num__assists_p90                      -0.0274
num__prev_share_tm                     0.0269
num__fouls_p90                        -0.0251
num__age                               0.0143
num__missingindicator_prev_share_tm   -0.0137
num__goals_p90                         0.0136
cat__position_Attack                  -0.0111


In [32]:
data.drop(columns=["variants"], errors="ignore").to_csv("transfers_joined.csv", index=False)
link.to_csv("wyscout_tm_link.csv", index=False)
results.to_csv("baseline_results.csv", index=False)
print("Saved transfers_joined.csv, wyscout_tm_link.csv, split.csv, baseline_results.csv")

Saved transfers_joined.csv, wyscout_tm_link.csv, split.csv, baseline_results.csv
